# Batch variant scoring

In this notebook, we demonstrate how to score many variants using AlphaGenome.

To prepare numerous variants for scoring, provide the following information as
columns in a tab-separated Variant Call Format (VCF) file: - `variant_id`: A
unique identifier for each variant. - `CHROM`: Chromosome, specified as a string
beginning with 'chr' (e.g., 'chr1'). - `POS`: Integer representing the base pair
position (1-based; build hg38 (human) or mm10 (mouse) - see
[FAQ](https://www.alphagenomedocs.com/faqs.html#how-do-i-define-a-variant)). -
`REF`: The reference nucleotide sequence at the specified position. - `ALT`: The
alternate nucleotide sequence at the specified position.

``` {tip}
Open this tutorial in Google Colab for interactive viewing.
```

In [6]:
# @title Install AlphaGenome

# @markdown Run this cell to install AlphaGenome.
from IPython.display import clear_output
! pip install alphagenome
clear_output()

In [7]:
# @title Setup and imports.

from io import StringIO
from alphagenome import colab_utils
from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers
from google.colab import data_table, files
import pandas as pd
from tqdm import tqdm

data_table.enable_dataframe_formatter()

# Load the model.
dna_model = dna_client.create(colab_utils.get_api_key())

In [12]:
vcf_file = '/content/AQP4_all_variants.vcf'  # @param

vcf = pd.read_csv(vcf_file, sep='\t')


vcf_1 = vcf.iloc[:1000,:]
vcf_2 = vcf.iloc[1000:,:]



,variant_id,CHROM,POS,REF,ALT
1000,1198136509,chr18,26862070,C,T
1001,1197967360,chr18,26859681,C,T
1002,1197957747,chr18,26852832,G,C
1003,1197753732,chr18,26866488,G,C
1004,1197477868,chr18,26853347,T,C
...,...,...,...,...,...
2477,162009,chr18,26864250,G,A
2478,162006,chr18,26866570,C,T
2479,162005,chr18,26867822,G,A
2480,151244,chr18,26866755,T,C


In [21]:
dir(dna_model.score_variant)


Help on method score_variant in module alphagenome.models.dna_client:

score_variant(interval: alphagenome.data.genome.Interval, variant: alphagenome.data.genome.Variant, variant_scorers: collections.abc.Sequence[~VariantScorerTypes] = (), *, organism: alphagenome.models.dna_model.Organism = <Organism.HOMO_SAPIENS: 9606>) -> list[anndata._core.anndata.AnnData] method of alphagenome.models.dna_client.DnaClient instance
    Generate variant scores for a single given DNA variant.

    Args:
      interval: DNA interval to make prediction for.
      variant: DNA variant to make prediction for.
      variant_scorers: Sequence of variant scorers to use for scoring. If no
        variant scorers are provided, the recommended variant scorers for the
        organism will be used.
      organism: Organism to use for the prediction.

    Returns:
      List of `AnnData` variant scores.



In [35]:
# @title Score batch of variants.

# Load VCF file containing variants.
vcf_file = '/content/AQP4_all_variants.vcf'  # @param

# We provide an example list of variants to illustrate.
#vcf_file = """variant_id\tCHROM\tPOS\tREF\tALT
#chr3_58394738_A_T_b38\tchr3\t58394738\tA\tT
#chr8_28520_G_C_b38\tchr8\t28520\tG\tC
#chr16_636337_G_A_b38\tchr16\t636337\tG\tA
#chr16_1135446_G_T_b38\tchr16\t1135446\tG\tT
#"""

vcf_full = pd.read_csv(vcf_file, sep='\t')

required_columns = ['variant_id', 'CHROM', 'POS', 'REF', 'ALT']
for column in required_columns:
  if column not in vcf_full.columns:
    raise ValueError(f'VCF file is missing required column: {column}.')

organism = 'human'  # @param ["human", "mouse"] {type:"string"}

# @markdown Specify length of sequence around variants to predict:
sequence_length = '1MB'  # @param ["16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]

# @markdown Specify which scorers to use to score your variants:
score_rna_seq = True  # @param { type: "boolean"}
score_cage = True  # @param { type: "boolean" }
score_procap = True  # @param { type: "boolean" }
score_atac = True  # @param { type: "boolean" }
score_dnase = True  # @param { type: "boolean" }
score_chip_histone = True  # @param { type: "boolean" }
score_chip_tf = True  # @param { type: "boolean" }
score_polyadenylation = True  # @param { type: "boolean" }
score_splice_sites = True  # @param { type: "boolean" }
score_splice_site_usage = True  # @param { type: "boolean" }
score_splice_junctions = True  # @param { type: "boolean" }

# @markdown Other settings:
download_predictions = True  # @param { type: "boolean" }

# Parse organism specification.
organism_map = {
    'human': dna_client.Organism.HOMO_SAPIENS,
    'mouse': dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[organism]

# Parse scorer specification.
scorer_selections = {
    'rna_seq': score_rna_seq,
    'cage': score_cage,
    'procap': score_procap,
    'atac': score_atac,
    'dnase': score_dnase,
    'chip_histone': score_chip_histone,
    'chip_tf': score_chip_tf,
    'polyadenylation': score_polyadenylation,
    'splice_sites': score_splice_sites,
    'splice_site_usage': score_splice_site_usage,
    'splice_junctions': score_splice_junctions,
}

all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
selected_scorers = [
    all_scorers[key]
    for key in all_scorers
    if scorer_selections.get(key.lower(), False)
]

# Remove any scorers or output types that are not supported for the chosen organism.
unsupported_scorers = [
    scorer
    for scorer in selected_scorers
    if (
        organism.value
        not in variant_scorers.SUPPORTED_ORGANISMS[scorer.base_variant_scorer]
    )
    | (
        (scorer.requested_output == dna_client.OutputType.PROCAP)
        & (organism == dna_client.Organism.MUS_MUSCULUS)
    )
]
if len(unsupported_scorers) > 0:
  print(
      f'Excluding {unsupported_scorers} scorers as they are not supported for'
      f' {organism}.'
  )
  for unsupported_scorer in unsupported_scorers:
    selected_scorers.remove(unsupported_scorer)



#if 'df_scores' not in locals() and 'df_scores' not in globals():
df_scores = pd.DataFrame()

chunk_size = 100  # number of rows per chunk
print(len(vcf_full)/chunk_size, "chunks")

for i in range(0, len(vcf_full), chunk_size):
  print("Processing chunk", i/chunk_size)
  vcf = vcf_full.iloc[i:i + chunk_size,:]

  # Score variants in the VCF file.
  results = []

  for i, vcf_row in tqdm(vcf.iterrows(), total=len(vcf)):
    variant = genome.Variant(
       chromosome=str(vcf_row.CHROM),
       position=int(vcf_row.POS),
       reference_bases=vcf_row.REF,
       alternate_bases=vcf_row.ALT,
       name=vcf_row.variant_id,
    )
    interval = variant.reference_interval.resize(sequence_length)

    variant_scores = dna_model.score_variant(
        interval=interval,
       variant=variant,
       variant_scorers=selected_scorers,
        organism=organism,
   )
    results.append(variant_scores)

  df_scores_part = variant_scorers.tidy_scores(results)
  df_scores_part = df_scores_part.loc[(df_scores_part['quantile_score'] > 0.90) | (df_scores_part['quantile_score'] < -0.90)]
  df_scores_part = df_scores_part.loc[(df_scores_part['biosample_name'] == 'astrocyte') | (df_scores_part['biosample_name'] == 'brain')]
  df_scores = pd.concat([df_scores, df_scores_part], ignore_index=True)

if download_predictions:
  df_scores.to_csv('variant_scores.csv', index=False)
  files.download('variant_scores.csv')

df_scores

24.82 chunks
Processing chunk 0


100%|██████████| 100/100 [02:04<00:00,  1.25s/it]


Processing chunk 100


100%|██████████| 100/100 [02:05<00:00,  1.25s/it]


Processing chunk 200


...

Processing chunk 2400


100%|██████████| 82/82 [01:40<00:00,  1.23s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,variant_id,scored_interval,gene_id,gene_name,gene_type,gene_strand,junction_Start,junction_End,output_type,variant_scorer,...,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,transcription_factor,histone_mark,gtex_tissue,raw_score,quantile_score
0,chr18:26857633:A>G,chr18:26333345-27381921:.,None,None,None,None,None,None,CHIP_TF,"CenterMaskScorer(requested_output=CHIP_TF, wid...",...,primary_cell,unknown,encode,single,False,EZH2,NaN,NaN,0.073365,0.939103
1,chr18:26857633:A>G,chr18:26333345-27381921:.,ENSG00000263677,ENSG00000263677,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,0.003267,0.920450
2,chr18:26857633:A>G,chr18:26333345-27381921:.,ENSG00000263846,CIAPIN1P,processed_pseudogene,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,0.002942,0.902982
3,chr18:26851957:C>T,chr18:26327669-27376245:.,ENSG00000260372,AQP4-AS1,lncRNA,+,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,-0.003446,-0.927208
4,chr18:26851957:C>T,chr18:26327669-27376245:.,ENSG00000265369,PCAT18,lncRNA,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,0.004594,0.959379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21000,chr18:26866755:T>C,chr18:26342467-27391043:.,None,None,None,None,None,None,CAGE,"CenterMaskScorer(requested_output=CAGE, width=...",...,tissue,NaN,fantom,NaN,NaN,NaN,NaN,NaN,0.086881,0.970426
21001,chr18:26855248:G>T,chr18:26330960-27379536:.,ENSG00000171885,AQP4,protein_coding,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,-0.004270,-0.949106
21002,chr18:26855248:G>T,chr18:26330960-27379536:.,ENSG00000265369,PCAT18,lncRNA,-,None,None,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),...,tissue,adult,encode,paired,False,NaN,NaN,,-0.004017,-0.941769
21003,chr18:26855248:G>T,chr18:26330960-27379536:.,ENSG00000171885,AQP4,protein_coding,-,None,None,SPLICE_SITE_USAGE,GeneMaskSplicingScorer(requested_output=SPLICE...,...,tissue,adult,encode,NaN,NaN,NaN,NaN,,0.007812,0.913199


Note that the resulting output dataframe could be quite large, especially if you
use many scorers for scoring. Very large dataframes can't be filtered
interactively, but you can interact with them using the standard `pandas`
commands:

In [36]:
# Examine just the effects of the variants on T-cells.
#columns = [c for c in df_scores.columns if c != 'ontology_curie']
#df_scores[(df_scores['ontology_curie'] == 'CL:0000127')][columns]

cols_of_interest = ['variant_id','gene_name','output_type','variant_scorer','ontology_curie','biosample_name','biosample_type','transcription_factor','histone_mark','raw_score','quantile_score']
df_scores[cols_of_interest]
df_scores = df_scores.loc[(df_scores['quantile_score'] > 0.95) | (df_scores['quantile_score'] < -0.95)]
df_scores = df_scores.loc[df_scores['biosample_name'] == 'astrocyte']
#df_scores = df_scores.loc[df_scores['biosample_name'] == 'brain']
df_scores[cols_of_interest]


,variant_id,gene_name,output_type,variant_scorer,ontology_curie,biosample_name,biosample_type,transcription_factor,histone_mark,raw_score,quantile_score
12,chr18:26861884:T>C,ENSG00000277534,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),CL:0000127,astrocyte,primary_cell,NaN,NaN,-0.003327,-0.952428
18,chr18:26862213:G>A,AQP4,SPLICE_JUNCTIONS,SpliceJunctionScorer(),CL:0000127,astrocyte,primary_cell,NaN,NaN,0.081360,0.988489
19,chr18:26862213:G>A,AQP4,SPLICE_JUNCTIONS,SpliceJunctionScorer(),CL:0000127,astrocyte,primary_cell,NaN,NaN,0.044647,0.978265
20,chr18:26862213:G>A,AQP4,SPLICE_JUNCTIONS,SpliceJunctionScorer(),CL:0000127,astrocyte,primary_cell,NaN,NaN,0.028625,0.958425
34,chr18:26852645:C>T,ENSG00000277534,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),CL:0000127,astrocyte,primary_cell,NaN,NaN,-0.003658,-0.962044
...,...,...,...,...,...,...,...,...,...,...,...
20970,chr18:26864250:G>A,None,CHIP_HISTONE,CenterMaskScorer(requested_output=CHIP_HISTONE...,CL:0000127,astrocyte,primary_cell,NaN,H3K9ac,-0.058095,-0.978501
20973,chr18:26864250:G>A,CIAPIN1P,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),CL:0000127,astrocyte,primary_cell,NaN,NaN,0.003519,0.957501
20976,chr18:26864250:G>A,AQP4,RNA_SEQ,GeneMaskLFCScorer(requested_output=RNA_SEQ),CL:0000127,astrocyte,primary_cell,NaN,NaN,-0.011767,-0.998540
20980,chr18:26864250:G>A,AQP4,SPLICE_JUNCTIONS,SpliceJunctionScorer(),CL:0000127,astrocyte,primary_cell,NaN,NaN,0.026901,0.952545
